In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# Output Files Description
# ============================================================================
# 1. eth_with_indicators.csv          - Original price data plus computed columns:
#                                       daily_return_pct, rolling_volatility_7d,
#                                       abs_daily_return.
# 2. data_verification_eth.csv        - Random sample verification of daily return
#                                       calculation accuracy.
# 3. volatility_anomaly_results_eth.csv - Full dataset with anomaly flags and
#                                         reasons appended.
# 4. multi_event_comparison_eth.csv   - Comparison table of key extreme dates
#                                       (max gain, max loss, max volatility).
# 5. volatility_analysis_report_eth.txt - Comprehensive English text report.
# ============================================================================

class VolatilityAnalyzer:
    """Volatility Analyzer - computes daily returns and rolling volatility,
       detects anomalies, and generates comparative reports."""

    def __init__(self, data_path='eth_etf_history.csv'):
        self.data_path = data_path
        self.df = None
        self.event_dates = {
            'max_gain_date': None,
            'max_loss_date': None,
            'max_volatility_date': None,
        }

    def load_and_preprocess(self):
        """Load data, compute daily returns and 7-day rolling volatility."""
        print("=" * 80)
        print("Data Loading and Preprocessing")
        print("=" * 80)

        # Read raw data
        self.df = pd.read_csv(self.data_path)
        self.df['date'] = pd.to_datetime(self.df['date'])
        self.df.set_index('date', inplace=True)
        self.df.sort_index(inplace=True)

        print(f"Data range: {self.df.index.min().date()} → {self.df.index.max().date()}")
        print(f"Total data points: {len(self.df)}")
        print(f"Original columns: {list(self.df.columns)}")

        # Compute daily percentage return
        self.df['daily_return_pct'] = self.df['price'].pct_change() * 100

        # Compute 7-day rolling volatility (standard deviation)
        self.df['rolling_volatility_7d'] = self.df['daily_return_pct'].rolling(window=7, min_periods=3).std()

        # Compute absolute daily return (used for ranking)
        self.df['abs_daily_return'] = self.df['daily_return_pct'].abs()

        print("\n✓ Added columns: daily_return_pct, rolling_volatility_7d, abs_daily_return")

        # Save dataset with indicators
        self.df.to_csv('eth_with_indicators.csv')
        print("✓ Data with indicators saved to: eth_with_indicators.csv")

        return self.df

    def verify_data_accuracy(self, sample_size=5):
        """Verify daily return calculation by comparing with manual computation."""
        print("\n" + "=" * 80)
        print("Data Accuracy Verification")
        print("=" * 80)

        # Select non-NaN return samples
        valid_indices = self.df['daily_return_pct'].dropna().index
        if len(valid_indices) == 0:
            print("No valid return data available for verification.")
            return

        sample_dates = np.random.choice(valid_indices, min(sample_size, len(valid_indices)), replace=False)
        results = []

        for date in sample_dates:
            # Convert numpy.datetime64 to pandas Timestamp to use strftime
            date_ts = pd.to_datetime(date)
            idx = self.df.index.get_loc(date_ts)
            if idx > 0:
                prev_price = self.df.iloc[idx-1]['price']
                cur_price = self.df.loc[date_ts, 'price']
                manual_return = (cur_price - prev_price) / prev_price * 100
                auto_return = self.df.loc[date_ts, 'daily_return_pct']
                return_match = np.isclose(manual_return, auto_return, rtol=1e-5)
            else:
                manual_return = None
                auto_return = self.df.loc[date_ts, 'daily_return_pct']
                return_match = pd.isna(auto_return)

            results.append({
                'Date': date_ts.strftime('%Y-%m-%d'),
                'Manual_Return': manual_return,
                'Auto_Return': auto_return,
                'Match': return_match
            })

        results_df = pd.DataFrame(results)
        print(results_df.to_string(index=False))
        results_df.to_csv('data_verification_eth.csv', index=False)
        print("\n✓ Verification results saved to: data_verification_eth.csv")
        return results_df

    def detect_anomalies(self, sigma_threshold=2, vol_multiple=3, percentile_cutoff=10):
        """Apply multiple anomaly detection criteria."""
        print("\n" + "=" * 80)
        print("Batch Anomaly Detection")
        print("=" * 80)

        # Historical statistics
        hist_returns = self.df['daily_return_pct'].dropna()
        hist_std = hist_returns.std()
        hist_avg_vol = self.df['rolling_volatility_7d'].dropna().mean()

        anomaly_flags = []
        for date, row in self.df.iterrows():
            ret_abs = abs(row['daily_return_pct']) if not pd.isna(row['daily_return_pct']) else 0
            # vol variable unused in tests but kept for clarity
            # vol = row['rolling_volatility_7d'] if not pd.isna(row['rolling_volatility_7d']) else 0

            test1 = ret_abs > sigma_threshold * hist_std
            test2 = ret_abs > vol_multiple * hist_avg_vol if hist_avg_vol > 0 else False

            rank = (hist_returns.abs() >= ret_abs).sum()
            total = len(hist_returns)
            percentile = rank / total * 100 if total > 0 else 100
            test3 = percentile <= percentile_cutoff

            is_anomaly = test1 or test2 or test3
            anomaly_flags.append(is_anomaly)

        self.df['is_anomaly'] = anomaly_flags
        self.df['anomaly_reason'] = ''

        for idx, date in enumerate(self.df.index):
            ret_abs = abs(self.df.loc[date, 'daily_return_pct']) if not pd.isna(self.df.loc[date, 'daily_return_pct']) else 0
            reasons = []
            if ret_abs > sigma_threshold * hist_std:
                reasons.append("2σ")
            if ret_abs > vol_multiple * hist_avg_vol:
                reasons.append("3x_volatility")
            rank = (hist_returns.abs() >= ret_abs).sum()
            percentile = rank / len(hist_returns) * 100
            if percentile <= percentile_cutoff:
                reasons.append(f"top_{percentile_cutoff}%")
            self.df.loc[date, 'anomaly_reason'] = ','.join(reasons) if reasons else ''

        anomaly_count = self.df['is_anomaly'].sum()
        total_count = len(self.df.dropna(subset=['daily_return_pct']))
        print(f"Valid trading days: {total_count}")
        print(f"Anomaly days detected: {anomaly_count} ({anomaly_count/total_count*100:.1f}%)")

        anomalies = self.df[self.df['is_anomaly']].nlargest(10, 'abs_daily_return')
        print("\nTop 10 most anomalous days (by absolute return):")
        print(anomalies[['price', 'daily_return_pct', 'rolling_volatility_7d', 'anomaly_reason']].to_string())

        self.df.to_csv('volatility_anomaly_results_eth.csv')
        print("\n✓ Anomaly detection results saved to: volatility_anomaly_results_eth.csv")
        return self.df

    def identify_key_dates(self):
        """Identify dates with maximum gain, loss, and absolute return."""
        if 'abs_daily_return' not in self.df.columns:
            self.df['abs_daily_return'] = self.df['daily_return_pct'].abs()

        max_gain_date = self.df['daily_return_pct'].idxmax()
        max_loss_date = self.df['daily_return_pct'].idxmin()
        max_vol_date = self.df['abs_daily_return'].idxmax()

        self.event_dates['max_gain_date'] = max_gain_date.strftime('%Y-%m-%d')
        self.event_dates['max_loss_date'] = max_loss_date.strftime('%Y-%m-%d')
        self.event_dates['max_volatility_date'] = max_vol_date.strftime('%Y-%m-%d')

        print("\nKey Date Identification:")
        print(f"Max gain day: {max_gain_date.date()} ({self.df.loc[max_gain_date, 'daily_return_pct']:.2f}%)")
        print(f"Max loss day: {max_loss_date.date()} ({self.df.loc[max_loss_date, 'daily_return_pct']:.2f}%)")
        print(f"Max volatility day (abs return): {max_vol_date.date()} ({self.df.loc[max_vol_date, 'abs_daily_return']:.2f}%)")
        return self.event_dates

    def multi_event_comparison(self):
        """Compare key dates using anomaly metrics."""
        print("\n" + "=" * 80)
        print("Multi-Event Comparison")
        print("=" * 80)

        if any(v is None for v in self.event_dates.values()):
            self.identify_key_dates()

        comparison = []
        for event_name, date_str in self.event_dates.items():
            date = pd.to_datetime(date_str)
            if date not in self.df.index:
                print(f"Warning: {event_name} date {date_str} not in dataset")
                continue

            row = self.df.loc[date]
            ret_abs = abs(row['daily_return_pct']) if not pd.isna(row['daily_return_pct']) else 0

            all_abs = self.df['daily_return_pct'].abs().dropna()
            rank = (all_abs >= ret_abs).sum()
            total = len(all_abs)
            percentile = rank / total * 100 if total > 0 else 0

            hist_avg_vol = self.df['rolling_volatility_7d'].dropna().mean()
            vol_multiple = row['rolling_volatility_7d'] / hist_avg_vol if hist_avg_vol > 0 and not pd.isna(row['rolling_volatility_7d']) else 0

            comparison.append({
                'Event': event_name,
                'Date': date_str,
                'Price': row['price'],
                'Daily_Return_%': row['daily_return_pct'],
                '7D_Volatility': row['rolling_volatility_7d'],
                'Volatility_Multiple': vol_multiple,
                'Percentile_Rank_%': percentile,
                'Is_Anomaly': 'Yes' if row.get('is_anomaly', False) else 'No',
                'Anomaly_Reasons': row.get('anomaly_reason', '')
            })

        comp_df = pd.DataFrame(comparison)
        print(comp_df.to_string(index=False))
        comp_df.to_csv('multi_event_comparison_eth.csv', index=False)
        print("\n✓ Multi-event comparison saved to: multi_event_comparison_eth.csv")
        return comp_df

    def generate_report(self):
        """Generate comprehensive English report."""
        report_lines = []
        report_lines.append("=" * 80)
        report_lines.append("ETH ETF Volatility Analysis Report")
        report_lines.append("=" * 80)
        report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report_lines.append(f"Data Range: {self.df.index.min().date()} to {self.df.index.max().date()}")
        report_lines.append("")

        anomaly_count = self.df['is_anomaly'].sum()
        total = len(self.df.dropna(subset=['daily_return_pct']))
        report_lines.append("I. Overall Anomaly Statistics")
        report_lines.append("-" * 40)
        report_lines.append(f"Valid trading days: {total}")
        report_lines.append(f"Anomaly days: {anomaly_count} ({anomaly_count/total*100:.1f}%)")

        reason_counts = self.df['anomaly_reason'].value_counts().to_dict()
        report_lines.append("\nAnomaly Reason Distribution:")
        for reason, cnt in reason_counts.items():
            if reason:
                report_lines.append(f"  {reason}: {cnt} days")

        report_lines.append("\nII. Key Date Comparison")
        report_lines.append("-" * 40)
        comp_df = self.multi_event_comparison()
        for _, row in comp_df.iterrows():
            report_lines.append(f"{row['Event']} ({row['Date']}):")
            report_lines.append(f"  Return: {row['Daily_Return_%']:.2f}%")
            report_lines.append(f"  Volatility multiple: {row['Volatility_Multiple']:.2f}x")
            report_lines.append(f"  Percentile rank: {row['Percentile_Rank_%']:.1f}%")
            report_lines.append(f"  Anomaly: {row['Is_Anomaly']} ({row['Anomaly_Reasons']})")
            report_lines.append("")

        report_content = "\n".join(report_lines)
        with open('volatility_analysis_report_eth.txt', 'w', encoding='utf-8') as f:
            f.write(report_content)
        print(report_content)
        print("\n✓ Comprehensive report saved to: volatility_analysis_report_eth.txt")
        return report_content

    def run_full_analysis(self):
        """Execute the full analysis pipeline."""
        self.load_and_preprocess()
        self.verify_data_accuracy()
        self.detect_anomalies()
        self.identify_key_dates()
        self.multi_event_comparison()
        self.generate_report()
        print("\n" + "=" * 80)
        print("ETH ETF Data Analysis Complete!")
        print("Generated files:")
        files = ['eth_with_indicators.csv', 'data_verification_eth.csv',
                 'volatility_anomaly_results_eth.csv', 'multi_event_comparison_eth.csv',
                 'volatility_analysis_report_eth.txt']
        for f in files:
            print(f"  - {f}")
        return True


# Main entry point
if __name__ == "__main__":
    # Ensure 'eth_etf_history.csv' is in the current working directory
    analyzer = VolatilityAnalyzer(data_path='eth_etf_history.csv')
    analyzer.run_full_analysis()

Data Loading and Preprocessing
Data range: 2024-01-10 → 2024-05-24
Total data points: 136
Original columns: ['price']

✓ Added columns: daily_return_pct, rolling_volatility_7d, abs_daily_return
✓ Data with indicators saved to: eth_with_indicators.csv

Data Accuracy Verification
      Date  Manual_Return  Auto_Return  Match
2024-02-18      -2.531646    -2.531646   True
2024-04-24      47.619048    47.619048   True
2024-01-22       0.000000     0.000000   True
2024-01-17     -17.272727   -17.272727   True
2024-04-12      -5.128205    -5.128205   True

✓ Verification results saved to: data_verification_eth.csv

Batch Anomaly Detection
Valid trading days: 135
Anomaly days detected: 13 (9.6%)

Top 10 most anomalous days (by absolute return):
            price  daily_return_pct  rolling_volatility_7d            anomaly_reason
date                                                                                
2024-05-21  0.610        542.105263             207.962293  2σ,3x_volatility,top_10